# Step 2: Study corridor along the Karakoram Highway (KKH)

**What this notebook does**
1. Asks OpenStreetMap for the Karakoram Highway only (one small, fast request)
2. Cuts the road at Khunjerab Pass (OpenStreetMap continues into China)
3. Converts to metres and builds a 10 km zone around the road (the paper used the same idea)
4. Saves two files you will use for the rest of the project

**How to run:** run the cells one by one, top to bottom (Shift+Enter). Read the printed output after each cell.

Map data: © OpenStreetMap contributors

## Cell 1: install extra libraries (run once)

In [ ]:
%pip install -q requests folium mapclassify geopandas matplotlib

## Cell 2: imports and settings
`NORTH` is where we cut the road near Khunjerab Pass. Change it here if needed.

In [ ]:
import os
import requests
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box, LineString

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# ---- Settings you can change ----
WEST, EAST   = 72.5, 75.6       # longitude (x)
SOUTH, NORTH = 33.7, 36.85      # latitude (y); 36.85 is about Khunjerab Pass
BUFFER_M     = 10_000           # 10 km in metres, same as the paper

## Cell 3: download only the Karakoram Highway
Instead of downloading every road and filtering later (very slow), we ask OpenStreetMap directly for roads named "Karakoram" or with reference N-35. This should take seconds to about a minute.

In [ ]:
QUERY = f"""
[out:json][timeout:120];
(
  way["highway"]["name"~"Karakoram",i]({SOUTH},{WEST},{NORTH},{EAST});
  way["highway"]["ref"~"^N[ -]?35$",i]({SOUTH},{WEST},{NORTH},{EAST});
);
out geom;
"""

SERVERS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]

data = None
for url in SERVERS:
    try:
        print("Trying", url, "...")
        r = requests.post(url, data={"data": QUERY}, timeout=150,
                          headers={"User-Agent": "kkh-landslide-student-project"})
        r.raise_for_status()
        data = r.json()
        break
    except Exception as e:
        print("  failed:", e)

assert data is not None, "Both servers failed. Wait a few minutes and run this cell again."

rows = []
for el in data["elements"]:
    if el["type"] == "way" and "geometry" in el:
        pts = [(p["lon"], p["lat"]) for p in el["geometry"]]
        if len(pts) >= 2:
            tags = el.get("tags", {})
            rows.append({"name": tags.get("name"), "ref": tags.get("ref"),
                         "geometry": LineString(pts)})

kkh = gpd.GeoDataFrame(rows, crs="EPSG:4326")
print("KKH pieces downloaded:", len(kkh))

## Cell 4: what did we get?
This shows the names and refs found. You should see "Karakoram Highway" and/or N-35.

In [ ]:
assert len(kkh) > 0, "Nothing found. Send me the output above and we will change the query."
print(kkh[["name", "ref"]].astype(str).value_counts().head(40))

## Cell 5: cut the road at Khunjerab Pass, then check it on a map
Zoom to the north end of the red line. It should stop near the pass and the border. If it goes into China, lower `NORTH` in Cell 2 (for example 36.82) and re-run from Cell 2. If it stops too early, raise it a little.

In [ ]:
clip_box = gpd.GeoSeries([box(WEST, SOUTH, EAST, NORTH)], crs="EPSG:4326")
kkh = gpd.clip(kkh, clip_box)

print("Pieces after cutting:", len(kkh))
print("Northernmost latitude of the road:", round(kkh.total_bounds[3], 3))

kkh.explore(tiles="OpenStreetMap", style_kwds={"weight": 4, "color": "red"})

## Cell 6: convert to metres and build the 10 km corridor

In [ ]:
kkh_m = kkh.to_crs("EPSG:32643")          # UTM zone 43N, units are metres
print("Total road length (km):", round(kkh_m.length.sum() / 1000))

# merge all pieces into one shape
try:
    line = kkh_m.geometry.union_all()
except AttributeError:
    line = kkh_m.geometry.unary_union

buffer_zone = gpd.GeoSeries([line.buffer(BUFFER_M)], crs="EPSG:32643")

# stop the corridor at the same northern cut so it does not reach into China
clip_box_m = clip_box.to_crs("EPSG:32643").iloc[0]
corridor = buffer_zone.intersection(clip_box_m)

print("Corridor area (km2):", round(corridor.area.sum() / 1e6))

## Cell 7: save the files and draw the result

In [ ]:
kkh_m.to_file("data/processed/kkh_line_pk.gpkg", driver="GPKG")
gpd.GeoDataFrame(geometry=corridor).to_file("data/processed/corridor_10km.gpkg", driver="GPKG")

fig, ax = plt.subplots(figsize=(7, 9))
corridor.plot(ax=ax, color="lightblue", edgecolor="steelblue")
kkh_m.plot(ax=ax, color="red", linewidth=1)
ax.set_title("KKH and 10 km corridor (UTM 43N, metres)")
plt.show()

## Cell 8: download your files (Google Colab only)
Colab deletes files when the session ends. On your own laptop you can skip this.

In [ ]:
try:
    from google.colab import files
    for f in ["kkh_line_pk.gpkg", "corridor_10km.gpkg"]:
        files.download("data/processed/" + f)
except ImportError:
    print("Not on Colab - the files are in the folder data/processed")